# 06 — Perturbation & Robustness Analysis (RO Dimension)

Analyses the three RO robustness test suites:
- **RO-1** Perturbation Sensitivity: 5,000 typographical perturbation tests
- **RO-3** OOD Detection: 3,000 out-of-distribution samples (10 categories)
- **RO-4** Semantic Invariance: 4,000 paraphrase pairs

Also reproduces the SF-3 ↔ RO-4 correlation (r=0.61) discussed in Section 6.2.


In [ ]:
import sys, os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

systems = ['S1','S2','S3','S4']

# Load perturbation data for all systems
typo, para, ood_all = [], [], []
for sid in systems:
    t = pd.read_csv(f'../data/perturbation_tests/{sid}/typographical_perturbations.csv')
    p = pd.read_csv(f'../data/perturbation_tests/{sid}/paraphrase_pairs.csv')
    o = pd.read_csv(f'../data/perturbation_tests/{sid}/ood_detection_samples.csv')
    t['system_id'] = sid; p['system_id'] = sid; o['system_id'] = sid
    typo.append(t); para.append(p); ood_all.append(o)

typo_df = pd.concat(typo, ignore_index=True)
para_df = pd.concat(para, ignore_index=True)
ood_df  = pd.concat(ood_all, ignore_index=True)

print(f"Typo tests: {len(typo_df):,}  |  Paraphrase pairs: {len(para_df):,}  |  OOD samples: {len(ood_df):,}")

Typo tests: 5,000  |  Paraphrase pairs: 4,000  |  OOD samples: 3,000


## 1. RO-1 Perturbation Sensitivity by type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sensitivity by perturbation type (all systems pooled)
type_sens = typo_df.groupby('perturbation_type')['sensitivity'].mean().sort_values(ascending=False)
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(type_sens)))
bars = axes[0].barh(type_sens.index, type_sens.values, color=colors, edgecolor='white')
axes[0].axvline(0.10, color='red', linestyle='--', linewidth=1.2, label='Threshold (0.10)')
axes[0].set_title('RO-1: Mean Sensitivity by Perturbation Type\n(5,000 tests, all systems)',fontsize=11)
axes[0].set_xlabel('Sensitivity (higher = worse)'); axes[0].legend()
axes[0].grid(axis='x', alpha=0.3)

# Sensitivity by system
sys_sens = typo_df.groupby('system_id')['sensitivity'].mean()
sys_colors = ['#4C72B0','#DD8452','#55A868','#C44E52']
axes[1].bar(sys_sens.index, sys_sens.values, color=sys_colors, edgecolor='white')
axes[1].axhline(0.10, color='red', linestyle='--', linewidth=1.2, label='Threshold (0.10)')
axes[1].set_title('RO-1: Mean Perturbation Sensitivity by System', fontsize=11)
axes[1].set_ylabel('Sensitivity'); axes[1].set_ylim(0, 0.20)
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('RO-1 Perturbation Sensitivity Analysis', fontsize=13)
plt.tight_layout()
plt.show()

print("\nRO-1 pass rate (sensitivity ≤ 0.10) by system:")
for sid in systems:
    sub = typo_df[typo_df['system_id']==sid]
    print(f"  {sid}: {sub['passes_ro1_threshold'].mean():.1%}  (mean sensitivity: {sub['sensitivity'].mean():.3f})")


RO-1 pass rate (sensitivity ≤ 0.10) by system:
  S1: 62.3%  (mean sensitivity: 0.092)
  S2: 74.1%  (mean sensitivity: 0.081)
  S3: 69.4%  (mean sensitivity: 0.086)
  S4: 80.2%  (mean sensitivity: 0.074)


## 2. RO-3 OOD Detection by category

In [ ]:
ood_cat = ood_df.groupby('ood_category')['correct_detection'].mean().sort_values()
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c' if v < 0.80 else '#2ecc71' for v in ood_cat.values]
ax.barh(ood_cat.index, ood_cat.values, color=colors, edgecolor='white')
ax.axvline(0.80, color='red', linestyle='--', linewidth=1.5, label='Threshold (0.80)')
ax.set_title('RO-3 OOD Detection Rate by Category  (3,000 samples, all systems)', fontsize=12)
ax.set_xlabel('Detection rate'); ax.set_xlim(0, 1.05)
ax.legend(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

overall = ood_df['correct_detection'].mean()
print(f"\nOverall OOD detection rate: {overall:.3f}  (threshold ≥ 0.80)")
print("Below-threshold categories:")
below = ood_cat[ood_cat < 0.80]
for cat, rate in below.items():
    print(f"  {cat}: {rate:.3f}")


Overall OOD detection rate: 0.821  (threshold ≥ 0.80)
Below-threshold categories:
  ambiguous: 0.763
  extremely_long_query: 0.779


## 3. RO-4 Semantic Invariance & SF-3 correlation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# RO-4 invariance distribution by system
for sid, color in zip(systems,['#4C72B0','#DD8452','#55A868','#C44E52']):
    sub = para_df[para_df['system_id']==sid]['semantic_invariance_score']
    axes[0].hist(sub, bins=20, alpha=0.6, label=sid, color=color, edgecolor='white')
axes[0].axvline(0.85, color='red', linestyle='--', linewidth=1.5, label='Threshold (0.85)')
axes[0].set_title('RO-4: Semantic Invariance Distribution', fontsize=11)
axes[0].set_xlabel('Semantic invariance score'); axes[0].legend()
axes[0].grid(alpha=0.2)

# SF-3 vs RO-4 scatter (system means)
daily = pd.read_csv('../data/raw/S1_Customer_Support_Chatbot/metric_snapshots/daily_metrics.csv')
ro4_sf3_data = []
for sid in systems:
    d = pd.read_csv(f'../data/raw/{sid}_{"Customer_Support_Chatbot" if sid=="S1" else "AI_Code_Assistant_IDE_Plugin" if sid=="S2" else "Document_Summarization_and_QA" if sid=="S3" else "Medical_Triage_Assistant"}/metric_snapshots/daily_metrics.csv')
    if 'RO-4' in d.columns and 'SF-3' in d.columns:
        for _, row in d.iterrows():
            ro4_sf3_data.append({'system_id':sid,'RO4':row.get('RO-4',np.nan),'SF3':row.get('SF-3',np.nan)})

rdf = pd.DataFrame(ro4_sf3_data).dropna()
if len(rdf) > 5:
    for sid, color in zip(systems,['#4C72B0','#DD8452','#55A868','#C44E52']):
        sub = rdf[rdf['system_id']==sid]
        axes[1].scatter(sub['RO4'], sub['SF3'], alpha=0.5, color=color, label=sid, s=25)
    corr = rdf[['RO4','SF3']].corr().iloc[0,1]
    axes[1].set_title(f'RO-4 vs SF-3 Correlation  (r={corr:.2f})', fontsize=11)
else:
    # Simulate from known correlation
    np.random.seed(42)
    n = 80
    ro4 = np.random.uniform(0.78, 0.99, n)
    sf3 = 4.0 - 3.0*ro4 + np.random.normal(0, 0.25, n)
    sf3 = np.clip(sf3, 0.4, 3.5)
    axes[1].scatter(ro4, sf3, alpha=0.5, color='steelblue', s=30)
    corr = np.corrcoef(ro4, sf3)[0,1]
    axes[1].set_title(f'RO-4 vs SF-3  (simulated, r≈{corr:.2f}, paper: r=0.61)', fontsize=11)

axes[1].set_xlabel('RO-4 Semantic Invariance'); axes[1].set_ylabel('SF-3 Hallucination Rate')
axes[1].grid(alpha=0.3)

plt.suptitle('RO-4 Invariance Analysis & SF-3 Correlation', fontsize=13)
plt.tight_layout()
plt.show()
print(f"Paper-reported r: 0.61  |  Practical implication: use RO-4 as cheap SF-3 proxy")

Paper-reported r: 0.61  |  Practical implication: use RO-4 as cheap SF-3 proxy
